# RMSNorm: Math Derivation, Inference Data Flow & CuTeDSL
*Root Mean Square Layer Normalization — from formula to warp shuffle.*

## 1. Where RMSNorm Sits During Inference

When a user types "What is the weather?", the system runs:
1. **Tokenizer** splits the string into token IDs (e.g. `[2, 1924, 374, 279, 9282, 30]`)
2. **Embedding lookup** maps each ID → a 4096-dimensional float vector → the initial hidden state `x`
3. **36 transformer layers** iteratively refine `x`, each firing multiple RMSNorms

Every RMSNorm call takes the same hidden state `x` (shape `[B, T, D]`) and re-scales it before a matrix multiply. Here is where each fires in a single decode step:

```
User prompt → tokenize → embed:   x₀ ∈ ℝ^{B×T×4096}
  ┌── Layer ℓ (×36) ──────────────────────────────────────────┐
  │  ① x_n = RMSNorm(x, γ_ln1)    [shape: B×T×4096]         │
  │  ② q   = x_n @ W_Q.T          [shape: B×T×4096]         │
  │  ③ k,v = x_n @ W_K.T / W_V.T  [shape: B×T×1024]         │
  │  ④ q   = RMSNorm(q, γ_qnorm)  [per head, shape: B×T×32×128] │
  │  ⑤ k   = RMSNorm(k, γ_knorm)  [per head, shape: B×T×8×128]  │
  │  ⑥ RoPE, attention, O-proj, residual add                  │
  │  ⑦ x_n = RMSNorm(x, γ_ln2)    [shape: B×T×4096]         │
  │  ⑧ gate, up, SwiGLU, down, residual add                  │
  └───────────────────────────────────────────────────────────┘
  Final: RMSNorm(x, γ_final) → lm_head → logits → next token
```

**Call count per token:**

| Norm | Count | Dim D |
|---|---|---|
| ln1 (pre-attention) | 36 | 4096 |
| q_norm (per-head Q) | 36 | 128 |
| k_norm (per-head K) | 36 | 128 |
| ln2 (pre-MLP) | 36 | 4096 |
| final (pre-lm_head) | 1 | 4096 |
| **Total** | **145** | — |

$$36 \times \text{ln1} + 36 \times \text{q\_norm} + 36 \times \text{k\_norm} + 36 \times \text{ln2} + 1 \times \text{final} = 145 \text{ calls/token}$$

At 145 calls per token, kernel launch overhead and memory bandwidth dominate — not the arithmetic inside each call.

## 2. Exact Input and Output Specification

| Field | Value |
|---|---|
| Input tensor | x ∈ ℝ^{B × T × D} |
| Input dtype | bfloat16 (2 bytes/element) |
| Input at decode | [1, 1, 4096] — 1 user, 1 new token |
| Input at prefill | [B, T, 4096] e.g. [32, 512, 4096] for 32 concurrent users |
| What produced input | Embedding lookup (layer 0) OR residual add `x += attn_out` (layers 1–35) |
| Parameters | γ ∈ ℝ^D: D=4096 for ln1/ln2/final, D=128 for q_norm/k_norm |
| Output | y ∈ ℝ^{B × T × D} — same shape as input |
| Output dtype | bfloat16 |
| What consumes output | Q/K/V projection after ln1, gate+up after ln2, attention QK dot-product after q_norm/k_norm |
| Compute shape | Row-wise: for each row of D values, compute ONE scalar (RMS), then broadcast back |

**Key insight:** RMSNorm treats the last dimension as a single unit. The `B×T` batch and time dimensions are entirely independent — no communication across tokens or batch items, only *within* a row of D elements. This is why one GPU block handles exactly one row.

## 3. The Formula

The RMSNorm operation for element $i$ of a row $x \in \mathbb{R}^D$ is:

$$\text{RMSNorm}(x, \gamma, \varepsilon)_i = \gamma_i \cdot \frac{x_i}{\text{RMS}(x)}$$

$$\text{RMS}(x) = \sqrt{\frac{1}{D}\sum_{j=1}^{D} x_j^2 + \varepsilon}$$

Substituting RMS inline to get the fully expanded single expression:

$$y_i = \gamma_i \cdot x_i \cdot \left(\frac{1}{\sqrt{\dfrac{1}{D}\displaystyle\sum_{j=1}^{D}x_j^2 + \varepsilon}}\right)$$

**The computation has exactly two phases:**

- **Phase 1 (reduce):** Read all $D$ values of $x$ → accumulate $\sum_j x_j^2$ → divide by $D$ → add $\varepsilon$ → take reciprocal square root. This collapses $D$ numbers to **one scalar** `inv_rms`. Threads must communicate with each other (via warp shuffle + shared memory) to complete this reduction.

- **Phase 2 (broadcast):** That single scalar `inv_rms` × each of the $D$ values × $\gamma_i$. **Fully parallel** — each thread handles its own elements independently, no cross-thread communication needed.

The two-phase structure is not just algorithmic; it maps directly to GPU hardware: phase 1 uses warp shuffle intrinsics and shared memory barriers, phase 2 uses independent FMAs.

## 4. Derivation: Why RMS, Not LayerNorm

**LayerNorm** (Ba et al., 2016) normalizes by mean and variance:

$$\text{LN}(x)_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} \cdot \gamma_i + \beta_i$$

where $\mu = \dfrac{1}{D}\displaystyle\sum_j x_j$ and $\sigma^2 = \dfrac{1}{D}\displaystyle\sum_j (x_j - \mu)^2$.

**Zhang & Sennrich (2019)** showed three terms can be dropped without quality loss:

**1. Drop $\mu$ (mean subtraction)**

The downstream projection $W$ has a bias $b$ that can absorb any constant shift in the distribution. Centering here (subtract $\mu$) then shifting back there (add $b$) is redundant. Removing $\mu$ eliminates one full sequential pass over all $D$ values.

**2. Drop $\sigma^2$ relative to $\mu$**

Once $\mu$ is gone, the variance $\dfrac{1}{D}\displaystyle\sum_j x_j^2$ *is* the mean-square — no separate computation needed. The denominator becomes simply $\sqrt{\text{ms}(x) + \varepsilon}$ where $\text{ms}(x) = \dfrac{1}{D}\displaystyle\sum_j x_j^2$.

**3. Drop $\beta$ (additive shift)**

Same redundancy as $\mu$ in reverse: the downstream bias handles shifting. Verified in Zhang & Sennrich (2019) on MT benchmarks — no measurable quality regression.

**Cost comparison:**

| Step | LayerNorm | RMSNorm |
|---|---|---|
| Pass 1 | compute $\mu = \text{mean}(x)$ | — |
| Pass 2 | compute $\sigma^2 = \text{mean}((x-\mu)^2)$ | compute $\text{ms} = \text{mean}(x^2)$ |
| Apply | $x_i = (x_i - \mu)/\sigma \cdot \gamma + \beta$ | $y_i = x_i / \text{rms} \cdot \gamma$ |
| HBM reads | 2× (two passes over x) | 1× |
| Extra ops | subtract $\mu$ from each $x_j$, add $\beta$ | none |
| Relative cost | 7–64% slower (empirical, model-size dependent) | baseline |

**Why this matters for inference:** At 145 RMSNorm calls per token and 380 GB/s HBM bandwidth (RTX 4080), every saved HBM pass is direct latency reduction. The single-pass nature of RMSNorm is what makes it viable at this call frequency.

## 5. Symbol Table

Every symbol used in the formula, Python implementation, CuTeDSL kernel, and raw CUDA/Triton code:

| Symbol | Definition | Python / PyTorch | CuTeDSL | CUDA / Triton |
|---|---|---|---|---|
| $x$ | Input activation matrix | `x: torch.Tensor` shape `[B,T,D]` | `mX: cute.Tensor` | `const __nv_bfloat16* x` / `x_ptr` |
| $D$ | Hidden dimension (row width) | `D = x.shape[-1]` | `mX.shape(1)` | `const int D` / `tl.constexpr` |
| $\gamma$ | Learned scale weight | `self.weight` (nn.Parameter, shape [D]) | `mW: cute.Tensor` | `const float* gamma` / `gamma_ptr` |
| $\varepsilon$ | Epsilon for numeric stability | `eps=1e-6` | `eps: float` | `const float eps` |
| $\sum_j x_j^2$ | Sum of squares (per row) | `(x**2).sum(-1, keepdim=True)` | loop: `ss += xi*xi` | `ss += xi*xi` / `tl.sum(xi*xi)` |
| $\text{ms}$ | Mean of squares | `ss / D` | `ss / D` | `ss / D` |
| $\text{inv\_rms}$ | $1/\text{RMS}(x)$ | `(ms + eps).rsqrt()` | `cute.rsqrt(ms + eps)` | `rsqrtf(ms + eps)` |
| $y$ | Output (normalized+scaled) | `x * inv_rms * gamma` | `mOut[row,i] = xi * inv_rms * wi` | `y[i] = xi * inv_rms * gi` |
| BLOCK | Threads per block | `128` (typical) | `cute.block_dim('x')` | `blockDim.x` / `BLOCK_SIZE: tl.constexpr` |
| smem | Shared memory reduction buffer | N/A | `smem = cute.shared_memory(BLOCK)` | `__shared__ float smem[BLOCK]` |

**Numeric precision note:** $x$ is stored as BF16 (2 bytes/element) but the sum-of-squares accumulation must be done in **FP32** to avoid catastrophic cancellation. BF16 has only 7 mantissa bits — squaring two large BF16 values and adding D=4096 of them would lose precision. All production implementations (FlashInfer, vLLM, quack) upcast to FP32 before accumulating $\sum x_j^2$.

In [ ]:
import torch, math

torch.manual_seed(0)
B, T, D = 2, 4, 4096
x     = torch.randn(B, T, D, dtype=torch.float32)
gamma = torch.ones(D)
eps   = 1e-6

# ── Level 0: high-level intent ─────────────────────────────────────
# y = RMSNorm(x, gamma, eps)

# ── Level 1: separate normalization from scaling ────────────────────
# y = normalize(x, eps) * gamma

# ── Level 2: what normalize means ───────────────────────────────────
# y = (x / rms(x)) * gamma

# ── Level 3: expand rms ─────────────────────────────────────────────
# y = (x / sqrt(mean_sq(x) + eps)) * gamma

# ── Level 4: expand mean_sq ─────────────────────────────────────────
# y = x / sqrt( (x**2).sum(-1, keepdim=True) / D + eps ) * gamma

# ── Level 5: every operation named explicitly ────────────────────────
ss      = (x ** 2).sum(dim=-1, keepdim=True)      # [B,T,1]  sum of squares
ms      = ss / D                                    # [B,T,1]  mean of squares
inv_rms = (ms + eps).rsqrt()                       # [B,T,1]  1/RMS  (rsqrt = 1/sqrt)
y       = x * inv_rms * gamma                      # [B,T,D]  normalize + scale

# Reference check
y_ref   = torch.nn.functional.rms_norm(x, (D,), gamma, eps)
err     = (y - y_ref).abs().max().item()
print(f"x shape: {list(x.shape)}")
print(f"ss shape (per row): {list(ss.shape)}")
print(f"inv_rms range: [{inv_rms.min():.4f}, {inv_rms.max():.4f}]")
print(f"y shape: {list(y.shape)}")
print(f"Max error vs torch.nn.functional.rms_norm: {err:.2e}")
assert err < 1e-4, "mismatch"
print("✓ All levels produce the same result")

## 6. Data Flow at Batch Scale

**Prefill scenario:** B=32 concurrent users, T=512 tokens each

- Total rows (independent RMSNorm computations): $32 \times 512 = 16{,}384$
- Total elements per call: $16{,}384 \times 4{,}096 = 67\text{M elements}$
- Memory (BF16): $67\text{M} \times 2\text{ bytes} = 134\text{ MB input} + 134\text{ MB output} = 268\text{ MB}$

**Arithmetic intensity (roofline analysis):**

$$\text{AI} = \frac{\text{FLOPs}}{\text{bytes}} = \frac{D \times 5}{D \times 4} = \frac{5}{4} = 1.25 \text{ FLOP/byte}$$

*(5 FLOPs/element: 1 square + 1 accumulate in phase 1, 1 multiply by inv\_rms + 1 multiply by gamma + that's ~5 counting the rsqrt amortized.)*

RTX 4080 ridge point ≈ 151 FLOP/byte → **RMSNorm is 120× below the compute ceiling. It is purely memory-bound.**

**GPU launch geometry (ln1/ln2/final, D=4096):**

```
Grid:   (16384, 1, 1)   ← one block per row (B×T rows)
Block:  (128, 1, 1)     ← BLOCK=128 threads per block
Stride: D / BLOCK = 4096 / 128 = 32 elements per thread
```

Each thread owns elements `{tid, tid+128, tid+256, ..., tid+3968}` — 32 elements interleaved for coalesced HBM access.

**QK-norm (D=128, 32 Q-heads):**

- Total rows: $32 \times 512 \times 32 = 524{,}288$ (batch × tokens × heads)
- Each row: only 128 elements → one **warp** (32 threads), 4 elements/thread
- Launch: `<<<(524288, 1, 1), (32, 1, 1)>>>`
- Warp-level reduction only: no shared memory barriers needed, uses `__shfl_xor_sync` only

**Decode scenario (latency-critical path):**

- Each user: shape `[1, 1, 4096]` — just the single new token
- With B=32 users: 32 rows total → trivial arithmetic, ~10µs kernel time
- But **145 kernel launches** × ~10µs launch overhead = **1.45ms overhead per token**
- At 30 tokens/sec target, that's 4.4% of total token budget just in launch overhead
- **CUDA graphs** record all 145 launches into a single graph replay → reduces to ~0.1ms

**Memory bandwidth utilization per call (prefill, 268 MB):**

$$\text{Time}_{BW} = \frac{268 \text{ MB}}{380 \text{ GB/s}} = 0.70 \text{ ms per call}$$

At 145 calls/token: $0.70 \times 145 = 101\text{ ms}$ of pure bandwidth time — before any GEMM. This is why RMSNorm fusion with adjacent GEMMs (norm→GEMM epilogue) is an active research area.

In [ ]:
# This shows how the Python math maps to CuTeDSL primitives.
# Run in an environment with cutedsl installed (or read as pseudocode).

# import cutlass
# import cute

# The kernel below maps directly to the Level 5 Python above:
#
#   Python:  ss = (x**2).sum(-1, keepdim=True)
#   CuTeDSL: each thread accumulates a partial ss, then block_reduce
#
#   Python:  inv_rms = (ss/D + eps).rsqrt()
#   CuTeDSL: cute.rsqrt(smem[0] / D + eps)
#
#   Python:  y = x * inv_rms * gamma
#   CuTeDSL: mOut[row,i] = mX[row,i] * inv_rms * mW[i]

BLOCK = 128   # threads per block

# @cute.kernel
# def rmsnorm_sm89(mX, mW, mOut, eps: float):
#     row   = cute.block_idx('x')    # which row (token) this block handles
#     tid   = cute.thread_idx('x')   # 0..127
#     D     = mX.shape[1]            # 4096
#
#     # ── Phase 1: each thread accumulates partial sum-of-squares ─────
#     ss = cutlass.Float32(0.0)
#     for i in cutlass.range(tid, D, BLOCK):   # thread tid owns elements tid, tid+128, ...
#         xi = cutlass.Float32(mX[row, i])     # BF16 → FP32 upcast (automatic)
#         ss = ss + xi * xi                    # accumulate in FP32
#
#     # ── Phase 1b: reduce partial sums across all 128 threads ─────────
#     # Warp shuffle (5 rounds, ~20 cycles) then SMEM for inter-warp (2 barriers)
#     ss = cute.block_reduce_sum(ss)           # smem[0] now holds total ss
#
#     # ── Compute inv_rms from the reduced total ───────────────────────
#     inv_rms = cute.rsqrt(ss / D + eps)      # rsqrt = 1/sqrt, ~1 cycle on SM89
#
#     # ── Phase 2: each thread normalizes its own elements ─────────────
#     for i in cutlass.range(tid, D, BLOCK):
#         xi      = cutlass.Float32(mX[row, i])
#         wi      = cutlass.Float32(mW[i])
#         mOut[row, i] = cutlass.BFloat16(xi * inv_rms * wi)   # back to BF16

print("CuTeDSL kernel structure:")
print("  Phase 1: partial ss per thread  →  sum across BLOCK threads  →  one scalar")
print("  Between: rsqrt(scalar/D + eps)")
print("  Phase 2: element-wise multiply  →  no cross-thread communication")
print()
print("Key CuTeDSL primitives used:")
print("  cute.block_idx('x')     → blockIdx.x")
print("  cute.thread_idx('x')    → threadIdx.x")
print("  cutlass.range(tid,D,N)  → for(int i=tid; i<D; i+=N)")
print("  cutlass.Float32(bf16)   → __bfloat162float (upcast)")
print("  cute.rsqrt(x)           → __frsqrt_rn(x)")
print("  cute.block_reduce_sum() → warp shuffle + SMEM tree")
print()
print("Reduction detail (BLOCK=128 = 4 warps of 32 threads):")
print("  Within each warp: 5 rounds of __shfl_xor_sync  (~20 cycles)")
print("  Across 4 warps:   warp 0 collects from smem[0..3]  (2 __syncthreads)")
print("  Total reduction:  ~30-40 cycles on SM89")

## 7. Production Kernel Fusion

In a standard (unfused) pipeline, `residual_add` and `rmsnorm` are separate kernel launches:

**Without fusion (2 kernels, 5 HBM passes):**
```
residual_add: reads x [B,T,D] + reads attn_out [B,T,D] → writes x_new [B,T,D]  (3 passes)
rmsnorm:      reads x_new [B,T,D] → writes y [B,T,D]                             (2 passes)
Total: 5 HBM passes of [B,T,D]
```

**With fused_add_rmsnorm (1 kernel, 4 HBM passes):**
```
reads x + attn_out → computes x_new = x+attn_out in registers → computes rms in registers
→ writes x_new and y  (2 writes only, 2 reads)
Total: 4 HBM passes — saves 1 pass of [B,T,D]
At prefill B=1,T=2048: 1 pass = 2048×4096×2 = 16 MB saved per call × 72 fused calls = 1.15 GB
```

The `x_new = x + attn_out` never hits HBM; it lives in registers. The reduction over `x_new` for RMS happens using those same registers. This is the essence of kernel fusion: **keep data in registers/SMEM, avoid round-tripping to HBM between logically adjacent operations.**

**Production implementations:**

| Library | Function | Source file | Notes |
|---|---|---|---|
| FlashInfer | `fused_add_rmsnorm(input, residual, weight, eps)` | `csrc/norm.cuh` | In-place residual update, separate normed output |
| vLLM | `fused_add_rms_norm_kernel<<<>>>` | `csrc/layernorm_kernels.cu` | Used in `layers/layernorm.py::RMSNorm.forward_cuda` |
| SGLang | Calls FlashInfer directly | `srt/layers/layernorm.py` | `from flashinfer.norm import fused_add_rmsnorm` |
| TRT-LLM | RmsNormPlugin (standalone) + fused inside FusedMHA | `plugin/rmsnormPlugin.cu` | Pre-norm fused into attention kernel when possible |
| quack (this project) | `sm89_v2_smem_tree_reduce` | `transformer_arch/01_rmsnorm/` | Standalone kernel; fusion target is norm+GEMM epilogue |

**Next frontier:** norm+GEMM epilogue fusion — instead of writing `y` back to HBM and then reading it again for the Q/K/V projection GEMM, compute the GEMM's input tiles directly from the normalized values in shared memory. This eliminates the `y` write entirely for the ln1→Q/K/V and ln2→gate/up paths, saving another 134 MB per call at prefill scale.

## 8. What Q, K, V Are and Why RMSNorm Feeds Them

To close the loop: what happens immediately *after* each RMSNorm call?

**Q (Query) — "what am I looking for?"**

After ln1, the normalized hidden state $x_n$ is projected to Q:
$$Q = x_n \cdot W_Q^T \quad \text{shape: } [B, T, 32 \times 128] = [B, T, 4096]$$
Then reshaped to $[B, T, 32, 128]$ (32 heads, head\_dim=128). `q_norm` then applies RMSNorm to each head independently: the 128-element vector at `Q[b, t, h, :]` is normalized as a unit. Head-wise independence means 32 separate D=128 RMSNorm operations per token per layer.

**K (Key) — "what do I contain?"**

$$K = x_n \cdot W_K^T \quad \text{shape: } [B, T, 8 \times 128] = [B, T, 1024]$$
Reshaped to $[B, T, 8, 128]$. `k_norm` normalizes each of the 8 KV heads. Qwen3 uses **GQA (Grouped Query Attention)**: 8 K-heads are shared across $32/8 = 4$ Q-heads each, reducing the KV cache size by 4×.

**V (Value) — "what information do I carry?"**

$$V = x_n \cdot W_V^T \quad \text{shape: } [B, T, 8 \times 128] = [B, T, 1024]$$
**No QK-norm applied to V.** V is the content to be retrieved, not a key for matching — normalization would destroy the magnitude information that carries semantic meaning.

**Why RMSNorm precision matters for Q and K specifically:**

The attention score between query $q$ and key $k$ is:
$$\text{score} = \frac{q \cdot k^T}{\sqrt{d_k}}$$

If `q_norm` computes `inv_rms` with even 1% error, the Q vector is scaled by $1.01\times$. Since scores are quadratic in Q and K (both normalized), a 1% error in each propagates to ~2% error in scores. After softmax (which amplifies score differences exponentially), this can shift the attended token from the correct one to a wrong one. FP32 accumulation for $\sum x_j^2$ is not paranoia — it is load-bearing for model correctness.

## One-Sentence Takeaway

RMSNorm is the lightest per-call operation in the transformer (0.8 FLOP/byte, strongly memory-bound) but the most frequent (145 calls/token), making kernel launch overhead and residual-add fusion the dominant optimization levers — not the arithmetic itself.